In [ ]:
from pathlib import Path
import cv2
import dask.array as da
import tifffile as tf
import numpy as np 
from skimage.color import rgb2hed, hed2rgb
from dask.diagnostics import ProgressBar
import skimage
import pickle 
import matplotlib.pyplot as plt
from tqdm import tqdm 

In [ ]:
def register(reference_image, image_to_register, algorithm, coord=None, image_to_warp=None, warp_shape=None):
    if(algorithm == 'SIFT'):
        orb = cv2.SIFT_create(contrastThreshold=0.005, edgeThreshold=5, sigma=1.2)
    else:
        orb = cv2.ORB_create()
    
    if(coord is not None):
        (keypoints1, descriptors1) = orb.detectAndCompute(reference_image[coord[0]:coord[1], coord[2]:coord[3]], None)
        (keypoints2, descriptors2) = orb.detectAndCompute(image_to_register[coord[0]:coord[1], coord[2]:coord[3]], None)
    else:
        (keypoints1, descriptors1) = orb.detectAndCompute(reference_image, None)
        (keypoints2, descriptors2) = orb.detectAndCompute(image_to_register, None)

    if(algorithm == 'SIFT'):
        FLANN_INDEX_KDTREE = 1
        index_params = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
        search_params = dict(checks=50)
        flann = cv2.FlannBasedMatcher(index_params, search_params)

        matches = flann.knnMatch(descriptors1, descriptors2, k=2)
        good_matches = []
        for m, n in matches:
            if m.distance < 0.9 * n.distance:
                good_matches.append(m)
        points1 = np.float32([keypoints1[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
        points2 = np.float32([keypoints2[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)
    else:
        bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
        matches = bf.match(descriptors1, descriptors2)
        good_matches = sorted(matches, key=lambda x: x.distance)
        points1 = np.zeros((len(good_matches), 2), dtype=np.float32)
        points2 = np.zeros_like(points1)
        for i, match in enumerate(good_matches):
            points1[i, :] = keypoints1[match.queryIdx].pt
            points2[i, :] = keypoints2[match.trainIdx].pt

    homography, _ = cv2.findHomography(points1, points2, cv2.RANSAC, 10)
    print(homography.shape)
    if(warp_shape is None):
        if(image_to_warp is None):
            height, width = reference_image.shape[:2]
        else:
            height, width = image_to_warp.shape[:2]
    else:
        height, width = warp_shape
    if(image_to_warp is None):
        registered_image = skimage.transform.warp(image_to_register, np.array(homography), output_shape=(height, width)) 
    else:
        registered_image = skimage.transform.warp(image_to_warp, np.array(homography), output_shape=(height, width))

    return homography, registered_image


In [ ]:
# === registration method === 
def register_fluorescent_dapi_images(reference_image_key, DAPI_to_register, algorithm, registration_base_path, key, coord=None):
    with ProgressBar():
        reference_image_computed = DAPI_to_register[reference_image_key].compute()

    complete_registration_base_path = registration_base_path + '/' + algorithm + key

    DAPI_registered = {}
    H_for_registration = {}

    reference_image_computed = cv2.convertScaleAbs(reference_image_computed,  alpha=(255.0/65535.0)).astype(np.uint8)
    for cycle_channel, cycle_image in DAPI_to_register.items():
        with ProgressBar():    
            cycle_image_computed = cycle_image.compute()
        cycle_image_computed = cv2.convertScaleAbs(cycle_image_computed,  alpha=(255.0/65535.0)).astype(np.uint8)
        H, cycle_image_registered = register(reference_image_computed, cycle_image_computed, algorithm, coord)
        DAPI_registered[cycle_channel] = cycle_image_registered
        del cycle_image_registered
        H_for_registration[cycle_channel] = H
        del H
    
    for cycle_channel, cycle_image in DAPI_registered.items():
        cycle = cycle_channel.split('_')[0]
        channel = cycle_channel.split('_')[-1]
        cycle_key = cycle + '_' + channel
        print(cycle_key)
        tf.imwrite(complete_registration_base_path + '/' + cycle_key + '_registered.tif', cycle_image)

    del DAPI_to_register



    return DAPI_registered, H_for_registration

def register_fluorescent_non_dapi_images(reference_image_key, non_DAPI_to_register, H_for_registration, algorithm, registration_base_path, key):
    complete_registration_base_path = registration_base_path + '/' + algorithm + key

    non_DAPI_registered = {}

    for cycle_channel, cycle_image in non_DAPI_to_register.items():
        with ProgressBar():    
            cycle_image_computed = cycle_image.compute()
        cycle = cycle_channel.split('_')[0]
        cycle_key = cycle +  '_CH1'
        height, width = cycle_image_computed.shape[:2]
        cycle_image_registered = skimage.transform.warp(cycle_image_computed, np.array(H_for_registration[cycle_key]), output_shape=(height, width))
        non_DAPI_registered[cycle_channel] = cycle_image_registered
    
    for cycle_channel, cycle_image in non_DAPI_registered.items():
        cycle = cycle_channel.split('_')[0]
        channel = cycle_channel.split('_')[-1]
        cycle_key = cycle + '_' + channel
        print(cycle_key)
        tf.imwrite(complete_registration_base_path + '/' + cycle_key + '_registered.tif', cycle_image)

    with open(complete_registration_base_path + '/IF_H_for_registration.pickle', 'wb') as f: 
        pickle.dump(H_for_registration, f)
    
    return non_DAPI_registered

# for each algorithm, register images for a sample 
# for H&E, extract hematoxylin 

def register_color_images(DAPI_for_registration, stain_images_to_register, algorithm, registration_base_path, key, coord=None):
    complete_registration_base_path = registration_base_path + '/' + algorithm + key

    stain_images_registered = {}
    H_for_registration = {}
    color_scalebar_dict = {}
    for stain, stain_image in stain_images_to_register.items(): 
        with ProgressBar():
            stain_image_computed = stain_image.compute()
        if(stain == 'H&E'):
            
            stain_image_resized_computed = cv2.resize(stain_image_computed, (DAPI_for_registration.shape[1], DAPI_for_registration.shape[0]))
            color_scalebar_dict[stain] = 0.5319*(stain_image_computed.shape[1]/stain_image_resized_computed.shape[1])

            
            if(coord is not None):
                stain_hed = skimage.exposure.equalize_adapthist(stain_image_resized_computed.copy()[coord[0]:coord[1], coord[2]:coord[3]])
                stain_hed = rgb2hed(stain_hed)
                null = np.zeros_like(stain_hed[:, :, 0])
                stain_h = hed2rgb(np.stack((stain_hed[:, :, 0], null, null), axis=-1))
                stain_h = cv2.bitwise_not(((stain_h / np.max(stain_h)) * 255).astype(np.uint8))
                stain_h = cv2.cvtColor(stain_h, cv2.COLOR_RGB2GRAY)

                H, stain_registered = register(DAPI_for_registration[coord[0]:coord[1], coord[2]:coord[3]], stain_h, algorithm, image_to_warp=stain_image_resized_computed)
                
            else:
                stain_hed = skimage.exposure.equalize_adapthist(stain_image_resized_computed)
                stain_hed = rgb2hed(stain_hed)
                null = np.zeros_like(stain_hed[:, :, 0])
                stain_h = hed2rgb(np.stack((stain_hed[:, :, 0], null, null), axis=-1))
                stain_h = cv2.bitwise_not(((stain_h / np.max(stain_h)) * 255).astype(np.uint8))
                stain_h = cv2.cvtColor(stain_h, cv2.COLOR_RGB2GRAY)
                                       
                H, stain_registered = register(DAPI_for_registration, stain_h, algorithm, image_to_warp=stain_image_resized_computed)
                H_for_registration[stain] = H


            stain_images_registered[stain] = stain_registered
        
        else:
            reference = (stain_images_registered['H&E'] / np.max(stain_images_registered['H&E']) *255).astype(np.uint8)
            stain_image_resized_computed = cv2.resize(stain_image_computed, (reference.shape[1], reference.shape[0]))

            reference = cv2.cvtColor(reference, cv2.COLOR_RGB2GRAY)
            stain_image_resized_computed_gray = cv2.cvtColor(stain_image_resized_computed, cv2.COLOR_RGB2GRAY)

            H, stain_registered = register(reference.astype(np.uint8), stain_image_resized_computed_gray.astype(np.uint8), algorithm, coord, image_to_warp=stain_image_resized_computed)
            H_for_registration[stain] = H 
            stain_images_registered[stain] = stain_registered
    
    for stain, image in stain_images_registered.items():
        print(stain)
        tf.imwrite(complete_registration_base_path + '/' + stain + '_registered.tif', image)

    with open(complete_registration_base_path + '/color_H_for_registration.pickle', 'wb') as f: 
        pickle.dump(H_for_registration, f)

        
    return stain_images_registered, color_scalebar_dict

In [ ]:
# === registration method === 
def register_bleached_fluorescent_dapi_images(reference_image_key, DAPI_registered, bleached_DAPI_to_register, registration_base_path, key, algorithm):
    
    complete_registration_base_path = registration_base_path + '/' + algorithm + '/' + key

    bleached_DAPI_registered = {}
    H_for_registration = {}

    
    clahe = cv2.createCLAHE(
    clipLimit=2.0,       
    tileGridSize=(8, 8))

    for cycle_channel, cycle_images in bleached_DAPI_to_register.items():
        print(cycle_channel)

        if(cycle_channel != 'Cycle 1_CH1'):
            continue 
        bleached_DAPI_registered[cycle_channel] = {}
        H_for_registration[cycle_channel] = {}

        for region, cycle_image in cycle_images.items():
            
            cycle_image = cv2.convertScaleAbs(cycle_image,  alpha=(255.0/65535.0)).astype(np.uint8)
            reference_image = DAPI_registered[cycle_channel]
            reference_image = (np.clip(reference_image, 0, 1) * 255).astype(np.uint8)
            H, cycle_image_registered = register(reference_image, cycle_image, algorithm)
            bleached_DAPI_registered[cycle_channel][region] = cycle_image_registered
            del cycle_image_registered
            H_for_registration[cycle_channel][region] = H
            del H
    
    for cycle_channel, cycle_images in bleached_DAPI_registered.items():
        for region, cycle_image in cycle_images.items():
            cycle = cycle_channel.split('_')[0]
            channel = cycle_channel.split('_')[-1]
            cycle_key = cycle + '_' + channel + '_' + region
            print(cycle_key)
            tf.imwrite(complete_registration_base_path + '/' + cycle_key +'_bleached_registered.tif', cycle_image)

    del bleached_DAPI_to_register



    return bleached_DAPI_registered, H_for_registration,


In [ ]:
def register_bleached_non_dapi_images(reference_image_key, bleached_non_DAPI_to_register, bleached_DAPI_registered, H_for_registration, algorithm, registration_base_path, key):
    complete_registration_base_path = registration_base_path + '/' + algorithm + key

    bleached_non_DAPI_registered = {}

    for cycle_channel, cycle_images in bleached_non_DAPI_to_register.items():
        bleached_non_DAPI_registered[cycle_channel] = {}
        for region, cycle_image in cycle_images.items():
            
            cycle = cycle_channel.split('_')[0]
            cycle_key = cycle +  '_CH1'
            height, width = bleached_DAPI_registered[cycle_key][region].shape[:2]
            cycle_image_computed = cycle_image
            cycle_image_registered = skimage.transform.warp(cycle_image_computed, np.array(H_for_registration[cycle_key][region]), output_shape=(height, width))
            bleached_non_DAPI_registered[cycle_channel][region] = cycle_image_registered
    
    for cycle_channel, cycle_image in bleached_non_DAPI_registered.items():
        for region, cycle_image in cycle_images.items():
            cycle = cycle_channel.split('_')[0]
            channel = cycle_channel.split('_')[-1]
            cycle_key = cycle + '_' + channel + '_' + region
            print(cycle_key)
            tf.imwrite(complete_registration_base_path + '/' + cycle_key + '_bleached_registered.tif', cycle_image)

    with open(complete_registration_base_path + '/bleached_H_for_registration.pickle', 'wb') as f: 
        pickle.dump(H_for_registration, f)
    
    return bleached_non_DAPI_registered

In [ ]:
imaging_base_path = r"" # path to processed images 

In [ ]:
algorithms = ['SIFT', 'ORB']

In [ ]:
color_stains = ['H&E', 'PAS', 'Trichrome']

In [ ]:
key = "" # sample identifier

In [ ]:
roi_path = r"" # path to ROI dictionary

In [ ]:
registration_base_path =  r"" # path to registered images

In [ ]:
roi_exists = True

In [ ]:
if(roi_exists):
    with open(roi_path + '/roi_coord_dict.pickle', 'rb') as f: 
        roi = pickle.load(f)

In [ ]:
# === create folder for registered images == 
Path(registration_base_path).mkdir(exist_ok=True)
for algorithm in algorithms: 
    Path(registration_base_path + '/' + algorithm).mkdir(exist_ok=True)
    Path(registration_base_path + '/' + algorithm + '/' + key.split('/')[1]).mkdir(exist_ok=True)
    Path(registration_base_path + '/' + algorithm + key).mkdir(exist_ok=True)

In [ ]:
def resize_chunk(chunk, factor):
    return cv2.resize(chunk, None, fx=1/factor, fy=1/factor)

In [ ]:
def invert_chunk(chunk):
    return cv2.bitwise_not(chunk)

In [ ]:
def white_balance(chunk, white_patch_average, image_average):
    return (chunk * (white_patch_average/image_average)).clip(0, 255).astype((int)).astype(np.uint8)

In [ ]:
# === load fluorescent and RGB images === 
def load_fluorescent_dapi_images(factor_downsample):
    fluorescent_images_path = Path(imaging_base_path + '/IF')

    dapi_paths = [f for f in fluorescent_images_path.rglob('*.tif') if 'Bleach' not in str(f) and 'stitched_background_removed' in str(f) and 'CH1' in str(f) and key.split('/')[2] in str(f)]
    
    DAPI_to_register_list = [f for f in dapi_paths] 
    DAPI_to_register = {}
    IF_scalebar_dict = {}

    for path in tqdm(DAPI_to_register_list):
        cycle_index = str(path).find('Cycle')
        ch_index = str(path).find('CH')
        cycle = str(path)[cycle_index: cycle_index + len('Cycle')+2]
        channel = str(path)[ch_index: ch_index + len('CH')+1]
        #print(cycle + '_' + channel)
        img = da.from_array(tf.imread(path), chunks=(512, 512))
        
        img = da.pad(img, ((0, int(np.ceil(img.shape[0]/512)*512-img.shape[0])),  (0, int(np.ceil(img.shape[1]/512)*512-img.shape[1]))))
        img = img.rechunk((512, 512), balance=True)
        img = img.map_blocks(resize_chunk, factor_downsample, dtype=np.uint16, chunks=(int(512/factor_downsample), int(512/factor_downsample)))
        img = da.pad(img, ((0, int(np.ceil(img.shape[0]/512)*512-img.shape[0])),  (0, int(np.ceil(img.shape[1]/512)*512-img.shape[1]))))
        img = img.rechunk(chunks=(512, 512), balance=True)
        DAPI_to_register[cycle + '_' + channel] = img
        IF_scalebar_dict[cycle + '_' + channel] = 0.1771*factor_downsample
        del img
    
    return DAPI_to_register, IF_scalebar_dict
    
def load_fluorescent_non_dapi_images(factor_downsample, IF_scalebar_dict):
    
    fluorescent_images_path = Path(imaging_base_path + '/IF')
    non_dapi_paths = [f for f in fluorescent_images_path.rglob('*.tif') if 'Bleach' not in str(f) and 'stitched_background_removed' in str(f) and 'CH1' not in str(f) and key.split('/')[2] in str(f)]
    non_DAPI_to_register_list = [f for f in non_dapi_paths]
    non_DAPI_to_register = {}

    for path in tqdm(non_DAPI_to_register_list):
        cycle_index = str(path).find('Cycle')
        ch_index = str(path).find('CH')
        cycle = str(path)[cycle_index: cycle_index + len('Cycle')+2]
        channel = str(path)[ch_index: ch_index + len('CH')+1]
        img = da.from_array(tf.imread(path), chunks=(512, 512))

        img = da.pad(img, ((0, int(np.ceil(img.shape[0]/512)*512-img.shape[0])),  (0, int(np.ceil(img.shape[1]/512)*512-img.shape[1]))))
        img = img.rechunk((512, 512), balance=True)
        img = img.map_blocks(resize_chunk, factor_downsample, dtype=np.uint16, chunks=(int(512/factor_downsample), int(512/factor_downsample)))
        img = da.pad(img, ((0, int(np.ceil(img.shape[0]/512)*512-img.shape[0])),  (0, int(np.ceil(img.shape[1]/512)*512-img.shape[1]))))
        img = img.rechunk(chunks=(512, 512), balance=True)
        IF_scalebar_dict[cycle + '_' + channel] = 0.1771*factor_downsample
        non_DAPI_to_register[cycle + '_' + channel] = img
        del img 
    
    return non_DAPI_to_register, IF_scalebar_dict

def load_color_images():
    stain_images_to_register = {}
    for stain in color_stains:
        print(stain)
        if(stain == 'PAS'):
            stain_path = Path(imaging_base_path + '/' + stain + '_'.join(key.split(' ')))
        else:
            stain_path = Path(imaging_base_path + '/' + stain + key) 
        stain_img_path = [f for f in stain_path.rglob('*.tif') if 'Raw.ome' in str(f)]
        stain_img = da.rot90(da.from_array(tf.imread(stain_img_path[0]), chunks=(512, 512, 3)), k=1)
        stain_img = da.pad(stain_img, ((0, int(np.ceil(stain_img.shape[0]/512)*512-stain_img.shape[0])),  (0, int(np.ceil(stain_img.shape[1]/512)*512-stain_img.shape[1])), (0, 0)))
        stain_img = stain_img.rechunk(chunks=(512, 512, 3), balance=True)

        white_patch = stain_img[:1000, :1000].compute()
        white_patch_average = white_patch[0].mean()
        stain_img_average = da.mean(stain_img).compute()
        stain_img = stain_img.map_blocks(white_balance, white_patch_average, stain_img_average)

        stain_images_to_register[stain] = stain_img

    return stain_images_to_register  


In [ ]:
def load_bleached_fluorescent_images(factor_downsample):
    fluorescent_images_path = Path(imaging_base_path + '/IF')

    dapi_paths = [f for f in fluorescent_images_path.rglob('*.tif') if 'Bleach' in str(f) and 'stitched_background_removed' in str(f) and 'CH1' in str(f) and key.split('/')[2] in str(f) and 'CD20' not in str(f)]
    print(dapi_paths)
    
    DAPI_to_register_list = [f for f in dapi_paths] 
    DAPI_to_register = {}

    for path in tqdm(DAPI_to_register_list):
        cycle_index = str(path).find('Cycle')
        ch_index = str(path).find('CH')
        region_index = str(path).find('Region')
        cycle = str(path)[cycle_index: cycle_index + len('Cycle')+2]
        channel = str(path)[ch_index: ch_index + len('CH')+1]
        region = str(path)[region_index: region_index + len('Region')+2]
        img = resize_chunk(tf.imread(path), factor_downsample)

        if(DAPI_to_register.get(cycle + '_' + channel) !=None):
            DAPI_to_register[cycle + '_' + channel][region] = img
        else:
            DAPI_to_register[cycle + '_' + channel] = {}
            DAPI_to_register[cycle + '_' + channel][region] = img
        del img
    
    non_dapi_paths = [f for f in fluorescent_images_path.rglob('*.tif') if 'Bleach' in str(f) and 'stitched_background_removed' in str(f) and 'CD20' not in str(f) and 'CH1' not in str(f) and key.split('/')[2] in str(f)]
    non_DAPI_to_register_list = [f for f in non_dapi_paths]
    non_DAPI_to_register = {}

    for path in tqdm(non_DAPI_to_register_list):
        cycle_index = str(path).find('Cycle')
        ch_index = str(path).find('CH')
        region_index = str(path).find('Region')
        cycle = str(path)[cycle_index: cycle_index + len('Cycle')+2]
        channel = str(path)[ch_index: ch_index + len('CH')+1]
        region = str(path)[region_index: region_index + len('Region')+2]
        img = resize_chunk(tf.imread(path), factor_downsample)

        if(non_DAPI_to_register.get(cycle + '_' + channel) !=None):
            non_DAPI_to_register[cycle + '_' + channel][region] = img
        else:
            non_DAPI_to_register[cycle + '_' + channel] = {}
            non_DAPI_to_register[cycle + '_' + channel][region] = img
        del img 
    
    return DAPI_to_register, non_DAPI_to_register


In [ ]:
DAPI_to_register, IF_scalebar_dict = load_fluorescent_dapi_images(2)

In [ ]:
DAPI_registered_SIFT, H_SIFT = register_fluorescent_dapi_images('Cycle 1_CH1', DAPI_to_register, 'SIFT', registration_base_path, key)                                            

In [ ]:
DAPI_registered_SIFT = {}
non_DAPI_registered_SIFT = {}

complete_registration_path = registration_base_path + '/SIFT' + key
registered_dapi_image_paths = [f for f in Path(complete_registration_path).rglob('*.tif') if 'CH1' in str(f) and 'bleached' not in str(f)]
registered_non_dapi_image_paths = [f for f in Path(complete_registration_path).rglob('*.tif') if 'CH1' not in str(f) and 'bleached' not in str(f) and 'Cycle' in str(f)]

for path in tqdm(registered_dapi_image_paths):
    cycle_channel_key = '_'.join(str(path).split('\\')[-1].split('_')[:2])
    DAPI_registered_SIFT[cycle_channel_key] = tf.imread(path)

for path in tqdm(registered_non_dapi_image_paths):
    cycle_channel_key = '_'.join(str(path).split('\\')[-1].split('_')[:2])
    non_DAPI_registered_SIFT[cycle_channel_key] = tf.imread(path)

In [ ]:
non_DAPI_to_register, IF_scalebar_dict = load_fluorescent_non_dapi_images(2, IF_scalebar_dict)

In [ ]:
non_DAPI_registered_SIFT = register_fluorescent_non_dapi_images('Cycle 1_CH1', non_DAPI_to_register, H_SIFT, 'SIFT', registration_base_path, key)

In [ ]:
bleached_DAPI_to_register, bleached_non_DAPI_to_register = load_bleached_fluorescent_images(2)

In [ ]:
bleached_DAPI_registered, bleached_H_registration = register_bleached_fluorescent_dapi_images('Cycle 1_CH1', DAPI_registered_SIFT, bleached_DAPI_to_register, registration_base_path, key, 'SIFT')

In [ ]:
bleached_DAPI_registered = {}

bleached_non_DAPI_registered = {}


complete_registration_path = registration_base_path + '/SIFT' + key
registered_dapi_image_paths = [f for f in Path(complete_registration_path).rglob('*.tif') if 'CH1' in str(f) and 'bleached'  in str(f)]
registered_non_dapi_image_paths = [f for f in Path(complete_registration_path).rglob('*.tif') if 'CH1' not in str(f) and 'bleached'  in str(f)]

for path in tqdm(registered_dapi_image_paths):
    cycle_channel_key = '_'.join(str(path).split('\\')[-1].split('_')[:2])
    region_key = str(path).split('\\')[-1].split('_')[2]
    if(bleached_DAPI_registered.get(cycle_channel_key)==None):
        bleached_DAPI_registered[cycle_channel_key] = {}
    bleached_DAPI_registered[cycle_channel_key][region_key] = tf.imread(path) 

for path in tqdm(registered_non_dapi_image_paths):
    cycle_channel_key = '_'.join(str(path).split('\\')[-1].split('_')[:2])
    region_key = str(path).split('\\')[-1].split('_')[2]
    if(bleached_non_DAPI_registered.get(cycle_channel_key)==None):
        bleached_non_DAPI_registered[cycle_channel_key] = {}
    bleached_non_DAPI_registered[cycle_channel_key][region_key] = tf.imread(path)

In [ ]:
from skimage.filters import threshold_otsu
def return_bleached_coordinates(image):
    
    thresh = threshold_otsu(image)
    mask = image > thresh
    coords = np.column_stack(np.nonzero(mask))
    ymin, xmin = coords.min(axis=0)
    ymax, xmax = coords.max(axis=0)

    return (ymin, ymax+1, xmin, xmax+1)


In [ ]:
max_size = (1024, 1224)

In [ ]:
bleached_coordinates = {}
for cycle_channel, images in bleached_DAPI_registered.items():
    bleached_coordinates[cycle_channel] = {}
    for region, image in images.items():
        bleached_coordinates[cycle_channel][region] = return_bleached_coordinates(image)

In [ ]:

processed_bleached_coordinates = {}
for cycle_channel, coordinates in bleached_coordinates.items():
    processed_bleached_coordinates[cycle_channel] = {}
    for region, coordinate in coordinates.items():
        processed_bleached_coordinates[cycle_channel][region] = (coordinate[0], coordinate[0]+max_size[0], coordinate[2], coordinate[2]+max_size[1])

In [ ]:
complete_registration_base_path =  registration_base_path + '/SIFT' + key

In [ ]:
with open(complete_registration_base_path + '/bleached_coordinates.pickle', 'wb') as f: 
    pickle.dump(bleached_coordinates, f)

In [ ]:
with open(complete_registration_base_path + '/bleached_processed_coordinates.pickle', 'wb') as f: 
    pickle.dump(processed_bleached_coordinates, f)

In [ ]:
bleached_non_DAPI_registered = register_bleached_non_dapi_images('Cycle 1_CH1', bleached_non_DAPI_to_register, bleached_DAPI_registered, bleached_H_registration, 'SIFT', registration_base_path, key)

In [ ]:
threshold = {
    'Cycle 1_CH4':[0.1, 0.1, 0.1],
    'Cycle 2_CH2':[0.5, 0.4, 0.5],
    'Cycle 3_CH2':[0.3, 0.4, 0.3],
}

In [ ]:
ch = ['_CH4', '_CH2', '_CH2']

In [ ]:
normal_images = {}
bleached_images = {}

i = 0
for cycle_channel, coordinates in processed_bleached_coordinates.items():
    cycle = cycle_channel.split('_')[0]

    normal_images[cycle + ch[i]] = []
    bleached_images[cycle + ch[i]] = []

    for region, coordinate in coordinates.items(): 
        normal = non_DAPI_registered_SIFT[cycle + ch[i]].copy()[coordinate[0]:coordinate[1], coordinate[2]:coordinate[3]]
        normal_max = np.max(normal)
        bleached = bleached_non_DAPI_registered[cycle + ch[i]][region].copy()[coordinate[0]:coordinate[1], coordinate[2]:coordinate[3]]
        normal[normal<threshold[cycle + ch[i]][i]*normal_max] = 0
        bleached[bleached<threshold[cycle + ch[i]][i]*normal_max] = 0
        
        normal_images[cycle + ch[i]].append(normal)
        bleached_images[cycle + ch[i]].append(bleached)

    i+=1

In [ ]:
def split_image_into_four(img):
    h, w = img.shape[:2]
    h_mid, w_mid = h // 2, w // 2

    return [
        img[:h_mid, :w_mid],   # top-left
        img[:h_mid, w_mid:],   # top-right
        img[h_mid:, :w_mid],   # bottom-left
        img[h_mid:, w_mid:]    # bottom-right
    ]

def split_dict_images(d):
    result = {}

    for key, images in d.items():
        split_images = []
        for img in images:
            split_images.extend(split_image_into_four(img))
        result[key] = split_images

    return result

normal_images_split = split_dict_images(normal_images)
bleached_images_split = split_dict_images(bleached_images)

In [ ]:
normal_sum = {}
bleached_sum = {}

In [ ]:
for cycle_channel in normal_images_split.keys():
    normal_image_list = normal_images_split[cycle_channel]
    bleached_image_list = bleached_images_split[cycle_channel]

    if(cycle_channel == 'Cycle 1_CH4'):
        normal_image_list = normal_image_list[:-4]
        bleached_image_list = bleached_image_list[:-4]

    if(cycle_channel == 'Cycle 2_CH2'):
        normal_image_list = normal_image_list[4:]
        bleached_image_list = bleached_image_list[4:]

    normal_sum[cycle_channel] = []
    bleached_sum[cycle_channel] = []

    for i in range(len(normal_image_list)):
        normal_sum[cycle_channel].append(np.sum(normal_image_list[i]))
        bleached_sum[cycle_channel].append(np.sum(bleached_image_list[i]))


In [ ]:
import matplotlib 

cycle = 'Cycle 3'
region = 'Region 3'
coordinate = processed_bleached_coordinates[cycle + '_CH1'][region]
normal = non_DAPI_registered_SIFT[cycle + '_CH2'].copy()[coordinate[0]:coordinate[1], coordinate[2]:coordinate[3]] 
bleached = bleached_non_DAPI_registered[cycle + '_CH2'][region].copy()[coordinate[0]:coordinate[1], coordinate[2]:coordinate[3]]
normal_max = np.max(normal)
bleached_max = np.max(bleached)
normal[normal < threshold[cycle + '_CH2'][int(region.split(' ')[-1])-1]*normal_max] = 0
bleached[bleached < threshold[cycle + '_CH2'][int(region.split(' ')[-1])-1]*normal_max] = 0

plt.rcParams.update({
    "font.size": 15
})


fig, ax = plt.subplots(1, 2)
fig.set_figwidth(15)
im0 = ax[0].imshow(normal, vmax=normal_max)
ax[0].axis('off')
cbar1 = fig.colorbar(im0, ax=ax[0])
cbar1.locator = matplotlib.ticker.MaxNLocator(nbins=5)
cbar1.update_ticks()
im1 = ax[1].imshow(bleached, vmax=normal_max) 
ax[1].axis('off')
cbar2 = fig.colorbar(im1, ax=ax[1])
cbar2.locator = matplotlib.ticker.MaxNLocator(nbins=5)
cbar2.update_ticks()
plt.subplots_adjust(wspace=0.01)


In [ ]:
import seaborn as sns
from statannotations.Annotator import Annotator
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"font.size": 20})

xtick_labels = {
    "Cycle 1_CH4": "CD68",
    "Cycle 2_CH2": "PODXL",
    "Cycle 3_CH2": "PANCK"
}

records = []
for key in normal_sum.keys():
    for v in normal_sum[key]:
        records.append({"Marker": key, "Group": "Normal", "Value": v})
    for v in bleached_sum[key]:
        records.append({"Marker": key, "Group": "Bleach", "Value": v})

df = pd.DataFrame(records)

palette = {"Normal": "dodgerblue", "Bleach": "grey"}

markers = list(normal_sum.keys())

fig, axes = plt.subplots(len(markers), 1, figsize=(6, 10), sharex=False)

for ax, marker in zip(axes, markers):

    sub_df = df[df["Marker"] == marker]

    sns.barplot(
        data=sub_df,
        y="Group",
        x="Value",
        estimator=np.mean,
        errorbar="se",
        palette=palette,
        orient="h",
        ax=ax
    )

    pairs = [("Normal", "Bleach")]

    annotator = Annotator(
        ax,
        pairs,
        data=sub_df,
        y="Group",
        x="Value", 
        orient='h'
    )

    annotator.configure(
        test="Wilcoxon",
        text_format="star",
        loc="inside"
    )

    annotator.apply_and_annotate()

    ax.set_title(xtick_labels[marker])
    ax.set_ylabel("")
    ax.set_xlabel("Intensity Sum")

plt.tight_layout()
plt.show()